# 🔵 Week 12 Lab — SOLUTIONS
## K-Means Clustering & Unsupervised Discovery

**Objectives:**
- Understand the difference between supervised classification and unsupervised clustering
- Implement k-means and visualise its convergence step by step
- Test whether clustering can recover reach directions without labels
- Explore what happens when k is wrong — which directions merge?
- Use the elbow method and silhouette analysis to estimate k
- Discover what clustering *cannot* find (healthy vs impaired)
- Compare k-means with Gaussian Mixture Models (GMMs)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, confusion_matrix
)
from matplotlib.patches import Ellipse

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})

cmap8 = plt.cm.get_cmap('Set1', 8)

## Upload Data

Upload `week8_data.pkl` — the same reaching dataset from Weeks 8–11.

This file contains:
- 480 trials (8 directions × 3 speeds × 20 subjects)
- 80 neural features (cosine-tuned M1 neurons, introduced in Week 6)
- 6 EMG features (muscle activations, introduced in Week 4)
- Subject labels for leave-one-subject-out cross-validation

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload week8_data.pkl

In [ ]:
# Load data
with open('week8_data.pkl', 'rb') as f:
    D = pickle.load(f)

X_neural = D['neural_rates']   # (480, 80)
X_emg    = D['X_raw']          # (480, 6)
y_dir    = D['targets']        # 8 directions (0-7)
y_bin    = (D['labels'] == 'impaired').astype(int)  # 0=healthy, 1=impaired
subjects = D['subjects']       # 20 subjects

muscle_names = ['BIC', 'TRI', 'AD', 'PD', 'BRD', 'PRO']

sc_neural = StandardScaler().fit_transform(X_neural)
sc_emg = StandardScaler().fit_transform(X_emg)
X_pca_n = PCA(n_components=2).fit_transform(sc_neural)
X_pca_e = PCA(n_components=2).fit_transform(sc_emg)

dirs = np.unique(y_dir)
dir_labels = [f'{int(d*45)}°' for d in dirs]

print(f'Dataset: {X_neural.shape[0]} trials, {X_neural.shape[1]} neurons, '
      f'{X_emg.shape[1]} muscles, {len(np.unique(subjects))} subjects')
print(f'Directions: {len(dirs)}, Healthy: {np.sum(y_bin==0)}, Impaired: {np.sum(y_bin==1)}')

---
## Part 1: From Labels to No Labels 🟢

Before running any algorithm, let's see what the data looks like with and without labels.

### Exercise 1.1: The same data, two views

Plot the neural data in PCA space twice: once coloured by direction (supervised view) and once in grey (unsupervised view).

**Question:** Can you see the clusters in the grey panel? The 8 clusters form a ring — this is because of cosine tuning (Week 6). Clustering will try to recover these groups from the grey.

In [ ]:
# Exercise 1.1: Supervised vs unsupervised view
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# TODO: Panel A — scatter X_pca_n coloured by y_dir
# TODO: Panel B — scatter X_pca_n all in grey
# YOUR CODE HERE

---
## Part 2: The K-Means Algorithm 🟢

Watch k-means converge step by step on our neural data.

### Exercise 2.1: K-means convergence

Run k-means with k=8 on the neural data and visualise the result in PCA space. Then run it manually to watch the centroids move.

**Expected result:** The algorithm converges within ~10 iterations, and the 8 cluster assignments closely match the 8 reach directions.

In [ ]:
# Exercise 2.1: K-means convergence
# TODO: Run KMeans(n_clusters=8) on sc_neural and print n_iter_
# TODO: Implement manual k-means in PCA space:
#   1. Initialise 8 centroids (random positions in X_pca_n)
#   2. Loop: assign each trial to nearest centroid, update centroids
#   3. Stop when centroids don't change
# TODO: Plot 4 stages (init, 1st assignment, 3 iters, converged)
# YOUR CODE HERE

---
## Part 3: Can Clustering Recover Directions? 🟢

We use the **Adjusted Rand Index (ARI)** to compare cluster assignments with true direction labels. ARI = 1 means perfect match, ARI = 0 means random.

**Important:** ARI requires ground-truth labels to compute. It is an evaluation tool, not something you can use in a truly label-free analysis.

### Exercise 3.1: Neural vs EMG clustering

Apply k-means with k=8 to both neural and EMG features. Compare ARI values.

**Expected result:** Neural ARI ≈ 0.92 (nearly perfect), EMG ARI ≈ 0.54 (partial). Why the difference? 80 cosine-tuned neurons separate 8 directions better than 6 muscles.

In [ ]:
# Exercise 3.1: Neural vs EMG clustering
# TODO: Run KMeans(n_clusters=8) on sc_neural and sc_emg
# TODO: Compute ARI for each using adjusted_rand_score(y_dir, labels)
# TODO: Plot side-by-side scatter in PCA space, coloured by cluster
# YOUR CODE HERE

### Exercise 3.2: What happens at k=4?

Run k-means with k=4 on neural data. Which direction pairs merge?

**Expected result:** Adjacent directions merge: 0°+315°, 45°+90°, 135°+180°, 225°+270°. These are biomechanically sensible groupings.

**Bonus:** Compute ARI against the original 8 labels AND against the 4 merged-pair labels. What does the difference tell you?

In [ ]:
# Exercise 3.2: K-means at k=4 — which directions merge?
# TODO: Run KMeans(n_clusters=4) on sc_neural
# TODO: Use confusion_matrix(y_dir, km4.labels_) to find which directions share clusters
# TODO: Compute ARI against original 8 labels AND against 4 merged-pair labels
# Hint: create y_pairs by grouping adjacent directions (0+7, 1+2, 3+4, 5+6)
# TODO: Plot 1x2 figure: Panel A = k=8, Panel B = k=4 with merged direction labels
# YOUR CODE HERE

---
## Part 4: Choosing K 🟡

In a real unsupervised analysis, you don't know k in advance. Two heuristics try to estimate it from the data.

### Exercise 4.1: Elbow method and silhouette analysis

Sweep k from 2 to 12. Plot inertia (elbow) and silhouette score.

**Expected result:** The elbow plot shows no clear bend. Silhouette peaks at k=3, not the true k=8. Neither heuristic finds the right answer — choosing k requires domain knowledge.

In [ ]:
# Exercise 4.1: Elbow and silhouette
ks = range(2, 13)
inertias = []
sils = []

# TODO: For each k, run KMeans and record:
#   km.inertia_ (for elbow) and silhouette_score(sc_neural, km.labels_)
# TODO: Plot side-by-side: (A) inertia vs k, (B) silhouette vs k
# YOUR CODE HERE

---
## Part 5: What Clustering Cannot Find 🟡

Every supervised classifier from Weeks 5–11 could distinguish healthy from impaired. Can unsupervised clustering?

### Exercise 5.1: Clustering vs healthy/impaired

Two experiments:
- **(A)** Apply k-means with k=2 to all 480 trials. Do the clusters match healthy/impaired?
- **(B)** Cluster only the impaired subjects with k=4. Do the clusters reveal "impairment subtypes" — or just rediscover reach directions?

**Expected result:** (A) ARI ≈ 0 — complete failure. (B) Clusters correlate with direction (ARI ≈ 0.53) not patient identity (ARI ≈ 0). What looks like subtypes is just the direction structure reappearing.

In [ ]:
# Exercise 5.1: What clustering cannot find
# TODO Panel A: Run KMeans(n_clusters=2) on sc_neural
#   Compute ARI vs y_bin (healthy/impaired)
# TODO Panel B: Select only impaired trials (y_bin == 1)
#   Run KMeans(n_clusters=4) on impaired data only
#   Compute ARI vs direction labels AND vs subject labels
# TODO: Plot side-by-side: (A) k=2 all data, (B) k=4 impaired only
# YOUR CODE HERE

### Exercise 5.2: The variance hierarchy

Why does clustering find direction instead of impairment? Compare the between-direction variance with the between-group (healthy/impaired) variance.

**Expected result:** Direction variance is hundreds of times larger than impairment variance. K-means finds the biggest signal.

In [ ]:
# Exercise 5.2: Variance hierarchy
# TODO: Compute between-direction variance:
#   for each direction, compute mean feature vector, then variance of those means
# TODO: Compute between-group variance (healthy vs impaired means)
# TODO: Plot side-by-side bars on log scale
# YOUR CODE HERE

---
## Part 6: Gaussian Mixture Models 🔴

K-means makes hard assignments (each trial belongs to exactly one cluster). GMMs make soft assignments (each trial has a probability of belonging to each cluster).

**Connection to previous weeks:**
- Week 7 (Naive Bayes): NB modelled each class as a Gaussian — GMM does the same without labels
- Week 10 (LDA vs QDA): LDA = tied covariance, QDA = full covariance — GMM offers the same choice via `covariance_type`

### Exercise 6.1: K-means vs GMM

Compare k-means and GMM (k=8) on neural data in PCA space.

**Expected result:** Nearly identical ARI (≈0.92). Our cosine-tuned clusters are roughly spherical and equal-sized, so GMM's extra flexibility doesn't help.

In [ ]:
# Exercise 6.1: K-means vs GMM
# TODO: Run KMeans and GaussianMixture (n_components=8) on X_pca_n
# TODO: Compute ARI for each
# TODO: Plot k-means (hard assignments) vs GMM (soft — use predict_proba for alpha)
# YOUR CODE HERE

### Exercise 6.2: GMM covariance types

Try all four covariance types: spherical, diagonal, tied, full.
Plot the covariance ellipses for each.

**Connection to Week 10:** 'tied' = like LDA (one shared covariance), 'full' = like QDA (each cluster its own covariance).

In [ ]:
# Exercise 6.2: GMM covariance types
# TODO: For each covariance_type in ['spherical', 'diag', 'tied', 'full']:
#   1. Fit GaussianMixture, predict labels, compute ARI
#   2. Plot clusters with covariance ellipses
# Hint: use Ellipse from matplotlib.patches
# YOUR CODE HERE

### Exercise 6.3: When does GMM beat k-means?

Our cosine-tuned data has roughly spherical clusters, so GMM and k-means perform identically. But what if clusters had different shapes and sizes?

Generate a hypothetical dataset with 3 clusters: one tight and circular, one elongated and tilted, one medium. Compare k-means and GMM.

**Expected result:** K-means splits the elongated cluster (ARI ≈ 0.67). GMM respects each cluster's shape (ARI ≈ 0.97).

In [ ]:
# Exercise 6.3: Hypothetical — when GMM beats k-means
np.random.seed(42)
n = 150

# Generate 3 clusters with different shapes (code provided)
c1 = np.random.randn(n, 2) * 0.5 + np.array([-2, 0])  # tight
angle = np.pi / 4
rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
c2 = (rot @ np.column_stack([np.random.randn(n)*2.5, np.random.randn(n)*0.4]).T).T + [2, 0]
angle3 = -np.pi / 3
rot3 = np.array([[np.cos(angle3), -np.sin(angle3)], [np.sin(angle3), np.cos(angle3)]])
c3 = (rot3 @ np.column_stack([np.random.randn(n)*0.6, np.random.randn(n)*1.8]).T).T + [0, 3]
X_hyp = np.vstack([c1, c2, c3])
y_hyp = np.array([0]*n + [1]*n + [2]*n)

# TODO: Run KMeans(n_clusters=3) and GaussianMixture(n_components=3, covariance_type='full')
# TODO: Compute ARI for each
# TODO: Plot 1x3: (A) true labels, (B) k-means, (C) GMM with covariance ellipses
# YOUR CODE HERE

---
## 💭 Thought Exercise

A researcher records neural activity from 50 stroke survivors during reaching. She hypothesises there are 2–3 subtypes of motor impairment but has no labels.

1. She runs k-means with k=3 and finds three clusters. She calls them "impairment subtypes." Based on Exercise 5.1, what should she check before drawing this conclusion?

2. Her silhouette analysis peaks at k=2. Does this mean there are truly 2 subtypes? What did we learn in Exercise 4.1 about silhouette's limitations?

3. She has both neural and EMG recordings. Based on Exercise 3.1, which feature set would you recommend for clustering, and why?

4. A colleague suggests using LOSO cross-validation to evaluate the clustering. Explain why this does not apply to unsupervised methods.